# Part 3 - Notebook kiểm chứng kết quả solver

Notebook này dùng thư viện chuẩn (NumPy) để kiểm tra độ đúng của các hàm trong `Part3/solvers.py`.

Các nhóm kiểm tra chính:
- So sánh nghiệm với nghiệm tham chiếu từ NumPy.
- So sánh residual $||Ax-b||_2$.
- Đo thời gian chạy từng phương pháp.
- Kiểm tra hành vi hội tụ của Gauss-Seidel trên ma trận phù hợp và không phù hợp.

In [ ]:
# Cell 2: Import thư viện và chuẩn bị đường dẫn import module
import os
import sys
import time
import math
import numpy as np

# Nếu máy chưa cài pandas thì notebook vẫn chạy bình thường bằng cách in text.
try:
    import pandas as pd
    HAS_PANDAS = True
except Exception:
    HAS_PANDAS = False

cwd = os.getcwd()
project_root = cwd
if not os.path.isdir(os.path.join(project_root, "Part3")):
    project_root = os.path.abspath(os.path.join(project_root, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from Part3.solvers import run_all_solvers

np.set_printoptions(precision=8, suppress=True)
print("Project root:", project_root)
print("Pandas available:", HAS_PANDAS)

In [ ]:
# Cell 3: Hàm hỗ trợ tính sai số và tạo nghiệm tham chiếu
def l2_norm(v):
    """Tính chuẩn Euclid (L2) của vector."""
    v = np.asarray(v, dtype=float)
    return float(np.sqrt(np.dot(v, v)))


def residual_norm(A, x, b):
    """Tính residual ||Ax-b||_2 để đo mức độ thỏa phương trình."""
    A = np.asarray(A, dtype=float)
    x = np.asarray(x, dtype=float)
    b = np.asarray(b, dtype=float)
    return l2_norm(A @ x - b)


def numpy_reference_solution(A, b):
    """
    Trả về nghiệm tham chiếu từ NumPy.
    - Ưu tiên solve() cho hệ vuông khả nghịch.
    - Nếu singular thì dùng lstsq() để lấy nghiệm least-squares.
    """
    A_np = np.asarray(A, dtype=float)
    b_np = np.asarray(b, dtype=float)

    try:
        x_ref = np.linalg.solve(A_np, b_np)
        ref_method = "numpy.linalg.solve"
    except np.linalg.LinAlgError:
        x_ref, *_ = np.linalg.lstsq(A_np, b_np, rcond=None)
        ref_method = "numpy.linalg.lstsq"

    return x_ref, ref_method


def is_strictly_diagonally_dominant(A):
    """Kiểm tra chéo trội nghiêm ngặt theo hàng."""
    A = np.asarray(A, dtype=float)
    n = A.shape[0]
    for i in range(n):
        diag = abs(A[i, i])
        off = np.sum(np.abs(A[i])) - diag
        if diag <= off:
            return False
    return True

In [ ]:
# Cell 4: Định nghĩa bộ test cho phần 3
# Mục tiêu là có cả case dễ, case hội tụ tốt cho GS, case khó và case singular.
rng = np.random.default_rng(2026)

A_small = [
    [4.0, 1.0, 2.0],
    [3.0, 5.0, 1.0],
    [1.0, 1.0, 3.0],
]
b_small = [4.0, 7.0, 3.0]

# Case chéo trội để Gauss-Seidel có điều kiện hội tụ tốt
n_dom = 10
M_dom = rng.normal(size=(n_dom, n_dom))
for i in range(n_dom):
    M_dom[i, i] = np.sum(np.abs(M_dom[i])) + 2.0
A_dom = M_dom.tolist()
b_dom = rng.normal(size=n_dom).tolist()

# Case gần singular: tạo bằng tích U * diag(sigma) * V^T với sigma rất nhỏ ở cuối
n_ns = 8
U, _ = np.linalg.qr(rng.normal(size=(n_ns, n_ns)))
V, _ = np.linalg.qr(rng.normal(size=(n_ns, n_ns)))
sigma = np.array([10.0, 3.0, 1.0, 0.1, 0.01, 1e-3, 1e-5, 1e-8])
A_near_singular = (U @ np.diag(sigma) @ V.T).tolist()
b_near_singular = rng.normal(size=n_ns).tolist()

# Case singular rõ ràng
A_singular = [
    [1.0, 2.0, 3.0],
    [2.0, 4.0, 6.0],
    [1.0, 1.0, 1.0],
]
b_singular = [6.0, 12.0, 2.0]

cases = [
    {
        "name": "small_well_conditioned",
        "A": A_small,
        "b": b_small,
        "x0": None,
        "tol_seidel": 1e-10,
        "max_iter": 3000,
    },
    {
        "name": "strictly_diagonally_dominant",
        "A": A_dom,
        "b": b_dom,
        "x0": [0.0] * n_dom,
        "tol_seidel": 1e-10,
        "max_iter": 5000,
    },
    {
        "name": "near_singular",
        "A": A_near_singular,
        "b": b_near_singular,
        "x0": [0.0] * n_ns,
        "tol_seidel": 1e-10,
        "max_iter": 6000,
    },
    {
        "name": "singular",
        "A": A_singular,
        "b": b_singular,
        "x0": [0.0, 0.0, 0.0],
        "tol_seidel": 1e-10,
        "max_iter": 5000,
    },
]

for c in cases:
    A_np = np.asarray(c['A'], dtype=float)
    print(
        f"{c['name']:28s} shape={A_np.shape}, diag_dominant={is_strictly_diagonally_dominant(A_np)}"
    )

In [ ]:
# Cell 5: Chạy toàn bộ solver trên từng case và thu thập kết quả
records = []

for case in cases:
    case_name = case['name']
    A = case['A']
    b = case['b']

    # Nghiệm tham chiếu bằng NumPy
    x_ref, ref_method = numpy_reference_solution(A, b)
    ref_residual = residual_norm(A, x_ref, b)

    # Chạy solver tự cài đặt
    out = run_all_solvers(
        A,
        b,
        x0=case['x0'],
        tol_seidel=case['tol_seidel'],
        max_iter_seidel=case['max_iter'],
    )

    for method, result in out.items():
        success = bool(result['success'])
        residual = float(result['residual'])
        runtime = float(result['runtime_sec'])
        iterations = int(result['iterations'])

        # Nếu solver trả nghiệm rỗng thì đánh dấu sai khác vô cùng
        if success and len(result['x']) > 0:
            x = np.asarray(result['x'], dtype=float)
            diff_x = l2_norm(x - x_ref)
            verify_residual = residual_norm(A, x, b)
        else:
            diff_x = float('inf')
            verify_residual = float('inf')

        records.append({
            'case': case_name,
            'ref_method': ref_method,
            'ref_residual': ref_residual,
            'method': method,
            'success': success,
            'iterations': iterations,
            'runtime_sec': runtime,
            'residual_reported': residual,
            'residual_recomputed': verify_residual,
            'x_diff_vs_numpy': diff_x,
            'message': result['message'],
        })

print(f"Collected rows: {len(records)}")

In [ ]:
# Cell 6: Hiển thị kết quả tổng hợp (DataFrame nếu có pandas)
if HAS_PANDAS:
    df = pd.DataFrame(records)
    cols = [
        'case', 'method', 'success', 'iterations', 'runtime_sec',
        'residual_reported', 'residual_recomputed', 'x_diff_vs_numpy',
        'ref_method', 'ref_residual', 'message'
    ]
    display(df[cols].sort_values(['case', 'method']).reset_index(drop=True))
else:
    # Fallback khi không có pandas
    for row in records:
        print(
            f"case={row['case']:28s} method={row['method']:15s} success={row['success']} "
            f"iter={row['iterations']:5d} residual={row['residual_reported']:.3e} "
            f"x_diff={row['x_diff_vs_numpy']:.3e} time={row['runtime_sec']:.6f}s"
        )

In [ ]:
# Cell 7: Assert kiểm tra tự động mức độ đúng
# Quy ước chấm:
# - Với case không singular, các phương pháp direct (gauss/qr/svd) nếu success
#   thì phải có sai khác nghiệm nhỏ hơn ngưỡng.
# - Gauss-Seidel chỉ kiểm tra ngưỡng khi nó hội tụ.
non_singular_cases = {'small_well_conditioned', 'strictly_diagonally_dominant', 'near_singular'}
direct_methods = {'gauss', 'qr_householder', 'svd'}

tol_direct = 1e-6
tol_gs = 1e-5

n_checked = 0
for row in records:
    case_name = row['case']
    method = row['method']

    if case_name in non_singular_cases and method in direct_methods and row['success']:
        assert row['x_diff_vs_numpy'] < tol_direct, (
            f"FAIL: {case_name}/{method} diff={row['x_diff_vs_numpy']} >= {tol_direct}"
        )
        n_checked += 1

    if case_name in non_singular_cases and method == 'gauss_seidel' and row['success']:
        assert row['x_diff_vs_numpy'] < tol_gs, (
            f"FAIL: {case_name}/{method} diff={row['x_diff_vs_numpy']} >= {tol_gs}"
        )
        n_checked += 1

print(f"PASS: {n_checked} comparison checks satisfied.")

# Check correctness for Part3 solvers

This notebook validates the solvers in `Part3/solvers.py` by comparing outputs with NumPy references.

In [ ]:
import os
import sys
from math import sqrt
import numpy as np

# Make sure project root is on path whether notebook runs from root or Part3.
PROJECT_ROOT = os.getcwd()
if not os.path.isdir(os.path.join(PROJECT_ROOT, "Part3")):
    PROJECT_ROOT = os.path.abspath(os.path.join(PROJECT_ROOT, ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from Part3.solvers import run_all_solvers

np.set_printoptions(precision=8, suppress=True)

In [ ]:
def l2_norm(v):
    return sqrt(float(np.dot(v, v)))


def safe_numpy_reference(A, b):
    """
    Return a reference solution using NumPy.
    - If A is full rank square, use np.linalg.solve.
    - Otherwise use least-squares solution.
    """
    A_np = np.array(A, dtype=float)
    b_np = np.array(b, dtype=float)
    n = A_np.shape[0]

    try:
        x_ref = np.linalg.solve(A_np, b_np)
        ref_method = "numpy.linalg.solve"
    except np.linalg.LinAlgError:
        x_ref, *_ = np.linalg.lstsq(A_np, b_np, rcond=None)
        ref_method = "numpy.linalg.lstsq"

    residual_ref = l2_norm(A_np @ x_ref - b_np)
    return x_ref, residual_ref, ref_method


def evaluate_case(case_name, A, b, x0=None, tol_seidel=1e-10, max_iter_seidel=3000):
    print(f"\n===== {case_name} =====")

    x_ref, residual_ref, ref_method = safe_numpy_reference(A, b)
    print(f"Reference: {ref_method}")
    print(f"Reference residual ||Ax-b||_2 = {residual_ref:.3e}")

    results = run_all_solvers(
        A, b, x0=x0, tol_seidel=tol_seidel, max_iter_seidel=max_iter_seidel
    )

    summary = {}
    for method, out in results.items():
        success = out["success"]
        x = np.array(out["x"], dtype=float) if out["x"] else None

        if success and x is not None and x.shape == x_ref.shape:
            diff = l2_norm(x - x_ref)
        else:
            diff = float("inf")

        summary[method] = {
            "success": success,
            "iterations": out["iterations"],
            "runtime_sec": out["runtime_sec"],
            "residual": out["residual"],
            "diff_vs_numpy": diff,
            "message": out["message"],
        }

    for method, row in summary.items():
        print(
            f"- {method:14s} | success={str(row['success']):5s} "
            f"| iter={row['iterations']:4d} "
            f"| residual={row['residual']:.3e} "
            f"| ||x-x_ref||_2={row['diff_vs_numpy']:.3e}"
        )

    # Basic correctness checks for direct methods (Gauss, QR, SVD).
    direct_methods = ["gauss", "qr_householder", "svd"]
    for m in direct_methods:
        if summary[m]["success"]:
            assert summary[m]["diff_vs_numpy"] < 1e-6, (
                f"{m} deviates from NumPy too much: {summary[m]['diff_vs_numpy']}"
            )

    # For Gauss-Seidel, only check if it converges.
    gs = summary["gauss_seidel"]
    if gs["success"]:
        assert gs["diff_vs_numpy"] < 1e-5, (
            f"gauss_seidel converged but differs too much: {gs['diff_vs_numpy']}"
        )

    print("Status: PASS (all applicable assertions satisfied)")
    return summary

In [ ]:
# Case 1: Small well-conditioned system
A1 = [
    [4.0, 1.0, 2.0],
    [3.0, 5.0, 1.0],
    [1.0, 1.0, 3.0],
]
b1 = [4.0, 7.0, 3.0]
summary1 = evaluate_case("Case 1 (small well-conditioned)", A1, b1)

In [ ]:
# Case 2: Random strictly diagonally dominant system (better for Gauss-Seidel)
rng = np.random.default_rng(42)
n = 8
R = rng.normal(size=(n, n))
A2 = R.copy()
for i in range(n):
    A2[i, i] = np.sum(np.abs(A2[i])) + 1.0
b2 = rng.normal(size=n)

summary2 = evaluate_case(
    "Case 2 (random diagonally dominant)",
    A2.tolist(),
    b2.tolist(),
    x0=[0.0] * n,
    tol_seidel=1e-10,
    max_iter_seidel=5000,
)

In [ ]:
# Case 3: Singular system (for behavior check, especially SVD)
A3 = [
    [1.0, 2.0, 3.0],
    [2.0, 4.0, 6.0],
    [1.0, 1.0, 1.0],
]
b3 = [6.0, 12.0, 2.0]

# For singular systems, some direct methods can fail by design.
# We still print outputs to inspect behavior and compare available methods.
summary3 = evaluate_case(
    "Case 3 (singular matrix)",
    A3,
    b3,
    x0=[0.0, 0.0, 0.0],
    tol_seidel=1e-10,
    max_iter_seidel=5000,
)

In [ ]:
# Optional compact summary table without pandas
all_cases = {
    "case1": summary1,
    "case2": summary2,
    "case3": summary3,
}

for cname, csum in all_cases.items():
    print(f"\n--- {cname} ---")
    for method, row in csum.items():
        print(
            f"{method:14s} success={str(row['success']):5s} "
            f"residual={row['residual']:.3e} "
            f"diff={row['diff_vs_numpy']:.3e} "
            f"time={row['runtime_sec']:.6f}s"
        )